# Vector + Graph Agent

This notebook extends the simple agent by adding a tool that:
1. Searches documents using vector similarity
2. Traverses the graph to find related entities (companies, risks)

The agent can now answer questions about document content with graph context.

---

## Setup

Import required modules and configure connections.

In [ ]:
import sys
sys.path.insert(0, '..')

from strands import Agent, tool
from strands.models import BedrockModel
from neo4j_graphrag.retrievers import VectorCypherRetriever
from neo4j_graphrag.schema import get_schema

from config import get_neo4j_driver, get_embedder, aws_config

In [ ]:
driver = get_neo4j_driver()
driver.verify_connectivity()

embedder = get_embedder()

print(f"Connected to Neo4j!")
print(f"Embedder: {embedder.model_id}")

## Create Vector Cypher Retriever

Define a retrieval query that gets document content along with related companies and risk factors.

In [ ]:
# Retrieval query that traverses from chunks to companies and their risks
retrieval_query = """
MATCH (node)-[:FROM_DOCUMENT]-(doc:Document)-[:FILED]-(company:Company)
OPTIONAL MATCH (company)-[:FACES_RISK]->(risk:RiskFactor)
WITH node, score, company, collect(risk.name)[0..10] AS risks
WHERE score IS NOT NULL
RETURN 
    node.text AS text,
    score,
    {company: company.name, risks: risks} AS metadata
ORDER BY score DESC
"""

vector_retriever = VectorCypherRetriever(
    driver=driver,
    index_name="chunkEmbeddings",
    embedder=embedder,
    retrieval_query=retrieval_query,
)

print("Vector retriever created!")

## Define Agent Tools

Create two tools:
1. **get_graph_schema** - Returns the database schema
2. **retrieve_financial_documents** - Searches documents with graph context

In [ ]:
@tool
def get_graph_schema() -> str:
    """
    Get the schema of the graph database including node labels,
    relationship types, and properties.
    
    Use this tool when the user asks about:
    - What data is in the database
    - The structure of the graph
    - What questions can be answered
    
    Returns:
        A string describing the database schema.
    """
    return get_schema(driver)


@tool
def retrieve_financial_documents(query: str) -> str:
    """
    Find details about companies in their financial documents using semantic search.
    
    This tool searches SEC 10-K filing documents and returns relevant passages
    along with the associated company name and risk factors.
    
    Use this tool when the user asks about:
    - Company products, services, or business
    - Risk factors mentioned in filings
    - Specific company information
    - Financial document content
    
    Args:
        query: The search query to find relevant documents.
        
    Returns:
        Relevant document passages with company and risk context.
    """
    try:
        results = vector_retriever.search(query_text=query, top_k=3)
        if not results.items:
            return "No documents found matching the query."
        return "\n\n".join(str(item.content) for item in results.items)
    except Exception as e:
        return f"Error searching documents: {e}"


print("Tools defined: get_graph_schema, retrieve_financial_documents")

## Create the Agent

Create an agent with both tools and appropriate instructions.

In [ ]:
bedrock_model = BedrockModel(
    model_id=aws_config.bedrock_model_id,
    region_name=aws_config.region,
    temperature=0.3,
)

agent = Agent(
    model=bedrock_model,
    system_prompt="""
    You are a helpful assistant that answers questions about a Neo4j graph database
    containing SEC 10-K financial filings from major companies.
    
    You have two tools:
    1. get_graph_schema - Use when asked about database structure
    2. retrieve_financial_documents - Use when asked about document content,
       company products, risks, or specific company information
    
    Choose the appropriate tool based on the question. Use the tool's results
    to provide accurate, grounded answers.
    """,
    tools=[get_graph_schema, retrieve_financial_documents]
)

print("Agent created with vector search capability!")

## Test Document Search

Ask the agent questions that require searching the financial documents.

In [ ]:
query = "What risk factors are mentioned in Apple's financial documents?"

print(f"User: {query}\n")
print("Assistant:")

response = agent(query)
print(response)

In [ ]:
query = "What products does Microsoft mention in its financial documents?"

print(f"User: {query}\n")
print("Assistant:")

response = agent(query)
print(response)

In [ ]:
query = "Tell me about cybersecurity threats mentioned in the filings."

print(f"User: {query}\n")
print("Assistant:")

response = agent(query)
print(response)

## Test Tool Selection

The agent should select the schema tool for structural questions and the document tool for content questions.

In [ ]:
# This should use the schema tool
query = "What types of entities are stored in the database?"

print(f"User: {query}\n")
print("Assistant:")

response = agent(query)
print(response)

In [ ]:
# This should use the document search tool
query = "What does NVIDIA say about AI in their filings?"

print(f"User: {query}\n")
print("Assistant:")

response = agent(query)
print(response)

## Summary

In this notebook, you learned:

1. **VectorCypherRetriever** - Combining vector search with graph traversal
2. **Multiple tools** - Creating an agent with multiple capabilities
3. **Tool selection** - How the agent chooses the right tool
4. **Graph context** - Including related entities in search results

The agent now intelligently selects between:
- Schema exploration for structural questions
- Document search for content questions

In the next notebook, you'll add a Text2Cypher tool for answering specific fact-based questions.

---

**Next:** [Multi-Tool Text2Cypher Agent](03_text2cypher_agent.ipynb)

In [ ]:
# Cleanup
driver.close()
print("Connection closed.")